# Data Merge
>We will merge and treat all the preprocessed dataframes to det the final dataset

In [ ]:
import pandas as pd
import numpy as np

In [106]:
semeval_e_train = pd.read_csv('../data/raw/English/E-c/2018-E-c-En-train.txt', sep='\t')
semeval_e_test = pd.read_csv('../data/raw/English/E-c/2018-E-c-En-test-gold.txt', sep='\t')
semeval_e_dev = pd.read_csv('../data/raw/English/E-c/2018-E-c-En-dev.txt', sep='\t')
semeval_e = pd.concat([semeval_e_train, semeval_e_test, semeval_e_dev])
semeval_e.drop(['ID'], axis=1, inplace=True)
emotion_cols = semeval_e.columns.drop('Tweet')
for col in emotion_cols:
    semeval_e[col] = semeval_e[col].replace({1: col, 0: ""})
semeval_e["emociones_merged"] = semeval_e[emotion_cols].apply(
    lambda fila: [val for val in fila if val != ""][0] if sum(fila != "") == 1 
                 else [val for val in fila if val != ""],
    axis=1
)
semeval_e.drop(emotion_cols, axis=1, inplace=True)
semeval_e.rename(columns={
    'Tweet': 'text',
    'emociones_merged': 'emotion'
}, inplace=True)

In [107]:
crowdflower = pd.read_csv('../data/preprocessed/crowdflower.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
go_emotions = pd.read_csv('../data/preprocessed/goemotions_full_clean.csv', names=['text', 'emotion']).drop(0)
isear = pd.read_csv('../data/preprocessed/isear.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
kushagra = pd.read_csv('../data/preprocessed/kushagra.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
meld = pd.read_csv('../data/preprocessed/meld.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
meld_dya = pd.read_csv('../data/preprocessed/meld-dya.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
meld_emorynlp = pd.read_csv('../data/preprocessed/meld-emorynlp.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
reddit = pd.read_csv('../data/preprocessed/redditposts.csv', names=['text', 'emotion']).drop(0)
semeval_ei = pd.read_csv('../data/preprocessed/semeval-ei.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
twitter = pd.read_csv('../data/preprocessed/twitter-emotion.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)



In [108]:
full_data = pd.concat([crowdflower, isear, kushagra, 
                       meld, meld_dya, meld_emorynlp, reddit, semeval_ei, twitter])

In [109]:
full_data = full_data.drop_duplicates(subset=['text'])
full_data


,text,emotion
1,@tiffanylue i know i was listenin to bad habi...,empty
2,Layin n bed with a headache ughhhh...waitin o...,sadness
3,Funeral ceremony...gloomy friday...,sadness
4,wants to hang out with friends SOON!,enthusiasm
5,@dannycastillo We want to trade with someone w...,neutral
...,...,...
4676,@MiSSLiNDZO no. 😭 the last two we were out-bid...,sadness
4736,When you lose somebody close to your heart you...,sadness
4738,Shantosh: How crazy would it be to walk past a...,sadness
4755,"hmm somehow twitter feels really depressing, m...",sadness


In [110]:
full_data['emotion'].unique()

array(['empty', 'sadness', 'enthusiasm', 'neutral', 'worry', 'surprise',
       'love', 'fun', 'hate', 'happiness', 'boredom', 'relief', 'anger',
       'fear', 'joy', 'sad', 'suprise', 'disgust', 'Joyful', 'Neutral',
       'Powerful', 'Mad', 'Sad', 'Scared', 'Peaceful', 'happy', 'angry',
       'rant', 'afraid', 'funny', 'surprised'], dtype=object)

In [111]:
full_data.replace(['empty', 'sadness', 'Sad'], 'sad', inplace=True)
full_data.replace(['enthusiasm', 'fun', 'happiness', 'Joyful', 'happy', 'funny', 'love'],
                  'joy', inplace=True)
full_data.replace(['worry', 'Scared', 'afraid'], 'fear', inplace=True)
full_data.replace(['suprise', 'surprised'], 'surprise', inplace=True)
full_data.replace(['hate', 'disgust', 'Mad', 'rant', 'angry'], 'anger', inplace=True)
full_data.replace(['neutral', 'boredom', 'relief', 'Neutral', 'Powerful', 'Peaceful'], 
                  np.nan, inplace=True)
full_data.dropna(inplace=True)


In [112]:
full_data['emotion'].unique()

array(['sad', 'joy', 'fear', 'surprise', 'anger'], dtype=object)

In [113]:
full_data['emotion'].value_counts()

emotion
joy         186724
sad         129669
anger        63011
fear         57755
surprise     16899
Name: count, dtype: int64

In [114]:
full_data

,text,emotion
1,@tiffanylue i know i was listenin to bad habi...,sad
2,Layin n bed with a headache ughhhh...waitin o...,sad
3,Funeral ceremony...gloomy friday...,sad
4,wants to hang out with friends SOON!,joy
6,Re-pinging @ghostridah14: why didn't you go to...,fear
...,...,...
4676,@MiSSLiNDZO no. 😭 the last two we were out-bid...,sad
4736,When you lose somebody close to your heart you...,sad
4738,Shantosh: How crazy would it be to walk past a...,sad
4755,"hmm somehow twitter feels really depressing, m...",sad


In [115]:
full_data.to_csv("../data/processed/data-no-goemotions-semevale.csv", index=False)